# NLI · tiga kelas · review
Premise → hypothesis: entailment, neutral, contradiction. [IndoNLI](https://github.com/ir-nlp-csui/indonli) beranotasi manusia, dengan test_lay dan test_expert terpisah. Tidak mereduksi neutral menjadi contradiction. Lima pendekatan, ditambah audit hypothesis-only untuk mendeteksi shortcut.

## 00 · Konfigurasi

Ubah sumber/kolom/ukuran sampel di sini. Tidak ada grafik otomatis.

In [1]:
SEED,TOP=42,15
RESULT_NAME='hasil_nli3_review'
N_TRAIN,N_VALID,N_TEST_EACH,EDA_N=6000,1500,1000,3000
EPOCHS,PAIR_MAX_LEN=3,256

## 01 · Lingkungan & preprocessing teknis

Safe cleaning: encoding, NFC, HTML entity, invisible, whitespace. Pertahankan angka, negasi, case, entitas dan bahasa. Tanpa stemming/stopword removal pada encoder.

In [2]:
import os,sys,subprocess,time,json,re,html,unicodedata,random,gc
from pathlib import Path
from collections import Counter
assert Path('/kaggle/input').exists(),'Jalankan di Kaggle.'
os.environ.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',HF_HUB_DISABLE_PROGRESS_BARS='1')
subprocess.run([sys.executable,'-m','pip','install','-q','ftfy>=6.3,<7','rank-bm25>=0.2,<1','sentencepiece>=0.2,<1','langid==1.1.6'],check=True)
import numpy as np,pandas as pd,torch,ftfy
from IPython.display import display
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import f1_score,classification_report,confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import Ridge,LogisticRegression
from transformers import AutoTokenizer,AutoModel,AutoModelForSequenceClassification
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.set_num_threads(2)
assert torch.cuda.is_available(),'Aktifkan GPU Kaggle.'
OUT=Path('/kaggle/working')/RESULT_NAME;OUT.mkdir(exist_ok=True)
pd.set_option('display.max_columns',None,'display.max_colwidth',120)
START=time.perf_counter()

In [3]:
def show(title,table):
    print('\n'+title)
    display((table.to_frame() if isinstance(table,pd.Series) else table).round(4))

def distribution(values):
    n=values.value_counts(dropna=False)
    return pd.DataFrame({'n':n,'%':100*n/(len(values) or np.nan)})

def clean(value):
    if not isinstance(value,str): return ''
    repaired=ftfy.fix_text(value,unescape_html=False,uncurl_quotes=False,fix_latin_ligatures=False,
        fix_character_width=False,remove_control_chars=False,normalization=None)
    text=unicodedata.normalize('NFC',html.unescape(repaired))
    text=text.translate(dict.fromkeys(map(ord,'\ufeff\u200b\u2060\x00'),None))
    text=re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]',' ',text)
    return re.sub(r'\s+',' ',text).strip()

def key(value): return clean(value).casefold()
def cap(frame,n): return frame.sample(min(n,len(frame)),random_state=SEED).reset_index(drop=True) if n else frame.reset_index(drop=True)
def locate(folder,repo):
    matches=list(Path('/kaggle/input').glob('**/models/'+folder+'/config.json'))
    return str(matches[0].parent) if matches else repo
NOISE={'url':r'https?://|www\.', 'mention':r'@\w+','hashtag':r'#\w+','angka':r'\d',
 'emoji/simbol':r'[\U0001F300-\U0001FAFF\u2600-\u27BF]','huruf_ulang':r'([^\W\d_])\1{2,}',
 'baca_ulang':r'[!?.,]{2,}','html':r'</?[a-zA-Z][^>]*>','encoding':r'\ufffd|Ã.|Â.',
 'non_ascii':r'[^\x00-\x7f]','spasi_ulang':r' {2,}','invisible':r'[\u200b\ufeff]',
 'KAPITAL':r'(?-i:\b[A-Z]{3,}\b)','huruf_angka':r'\b(?=\w*[a-z])(?=\w*\d)\w+\b',
 'slang_contoh':r'\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b',
 'negasi':r'\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|not|never)\b'}

In [4]:
def text_eda(values,title):
    valid=values.where(values.map(lambda v:isinstance(v,str)),'').fillna('')
    show(title+' kualitas',pd.Series({'n':len(values),'null':values.isna().sum(),'kosong/spasi':valid.str.strip().eq('').sum(),
        'unik_terisi':valid[valid.str.strip().ne('')].nunique(),'duplikat_ekstra_terisi':valid[valid.str.strip().ne('')].duplicated().sum()}))
    show(title+' length',pd.DataFrame({'karakter':valid.str.len(),'kata':valid.str.split().str.len()}).describe(percentiles=[.01,.5,.9,.95,.99]).T)
    sample=valid[valid.str.strip().ne('')].sample(min(EDA_N,valid.str.strip().ne('').sum()),random_state=SEED)
    tokens=sample.str.casefold().str.findall(r'[^\W\d_]+')
    count=pd.Series(Counter(w for seq in tokens for w in seq),dtype='int64').sort_values(ascending=False)
    show(title+' vocabulary sampel',pd.Series({'n_sample':len(sample),'token':count.sum(),'unik':len(count),
        'hapax':count.eq(1).sum(),'hapax_%_tipe':100*count.eq(1).mean(),'hapax_%_token':100*count.eq(1).sum()/(count.sum() or np.nan)}))
    show(title+' top words',count.head(TOP).rename('n').to_frame().assign(pct=count.head(TOP)/(count.sum() or np.nan)*100))
    for size in [2,3]:
        grams=pd.Series(Counter(' '.join(seq[i:i+size]) for seq in tokens for i in range(len(seq)-size+1)),dtype='int64').sort_values(ascending=False)
        show(title+f' top {size}-gram',grams.head(TOP))
    flags=pd.DataFrame({name:valid.str.count(regex,flags=re.I).gt(0) for name,regex in NOISE.items()})
    show(title+' noise · % semua baris, dapat overlap',pd.DataFrame({'n':flags.sum(),'%':flags.mean()*100}))
    terms=count.index.to_series().astype('string')
    weird=terms.str.len().ge(20)|terms.str.count(r'([^\W\d_])\1{2,}').gt(0)
    show(title+' kandidat kata aneh, bukan pasti typo',count[weird].head(TOP))
    from langid.langid import LanguageIdentifier,model
    from threadpoolctl import threadpool_limits
    detector=LanguageIdentifier.from_modelstring(model,norm_probs=True)
    lang_sample=sample.sample(min(150,len(sample)),random_state=SEED)
    with threadpool_limits(limits=1):
        lang=lang_sample.map(lambda s:detector.classify(s[:3000]) if len(s.split())>=5 else ('tidak_pasti',np.nan))
    show(title+' estimasi bahasa · % sampel, bukan gold',distribution(lang.map(lambda p:p[0] if p[1]>=.8 else 'tidak_pasti')))

In [5]:
def prepare_pairs(raw,a,b,target):
    show('Missing sebelum drop',pd.DataFrame({'n':raw.isna().sum(),'%':raw.isna().mean()*100}))
    frame=raw.copy();frame['a']=frame[a].map(clean);frame['b']=frame[b].map(clean)
    frame['y']=pd.to_numeric(frame[target],errors='coerce')
    ok=frame.a.ne('')&frame.b.ne('')&frame.y.notna()&np.isfinite(frame.y)
    print('Dibuang invalid:',int((~ok).sum()))
    return frame.loc[ok].reset_index(drop=True)

def endpoints(frame): return set(frame.a.map(key))|set(frame.b.map(key))
def purge(frame,blocked):
    keep=~frame.a.map(key).isin(blocked)&~frame.b.map(key).isin(blocked)
    print('Pasangan dikeluarkan karena teks lintas split:',int((~keep).sum()))
    return frame.loc[keep].reset_index(drop=True)

def pair_groups(frame):
    parent={}
    def root(x):
        parent.setdefault(x,x)
        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]
        return x
    for a,b in zip(frame.a.map(key),frame.b.map(key)):parent[root(a)]=root(b)
    return pd.factorize([root(x) for x in frame.a.map(key)])[0]

def pair_features(frame):
    rows=[]
    for a,b in zip(frame.a,frame.b):
        x,y=set(a.casefold().split()),set(b.casefold().split());union=x|y
        rows.append([len(x&y)/max(1,len(union)),len(x-y)/max(1,len(x)),len(y-x)/max(1,len(y)),len(a)/max(1,len(b)),
            int(bool(re.search(NOISE['negasi'],a,re.I)))!=int(bool(re.search(NOISE['negasi'],b,re.I))),
            len(set(re.findall(r'\d+',a))^set(re.findall(r'\d+',b)))])
    return np.asarray(rows,dtype='float32')

In [6]:
_encoder=None;_encoder_tok=None
@torch.inference_mode()
def embed(texts,prefix='query: '):
    global _encoder,_encoder_tok
    if _encoder is None:
        path=locate('e5','intfloat/multilingual-e5-base')
        _encoder_tok=AutoTokenizer.from_pretrained(path)
        _encoder=AutoModel.from_pretrained(path).cuda().eval()
    texts=list(texts);output=[]
    for start in range(0,len(texts),32):
        batch=_encoder_tok([prefix+t for t in texts[start:start+32]],padding=True,truncation=True,max_length=256,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):hidden=_encoder(**batch).last_hidden_state
        mask=batch.attention_mask.unsqueeze(-1)
        pooled=(hidden.float()*mask).sum(1)/mask.sum(1).clamp(min=1)
        output.append(torch.nn.functional.normalize(pooled,dim=1).cpu().numpy())
    return np.concatenate(output)

def release_encoder():
    global _encoder,_encoder_tok
    _encoder=None;_encoder_tok=None;gc.collect();torch.cuda.empty_cache()

In [7]:
def fine_tune_pairs(train,val,test,regression=False,label_names=('entailment','neutral','contradiction')):
    from datasets import Dataset,Value,disable_progress_bars
    from transformers import Trainer,TrainingArguments,DataCollatorWithPadding,set_seed
    from scipy.stats import spearmanr
    disable_progress_bars();set_seed(SEED);release_encoder()
    path=locate('indobert','indobenchmark/indobert-base-p1');tok=AutoTokenizer.from_pretrained(path)
    lengths=pd.Series([len(seq) for seq in tok(train.a.tolist(),train.b.tolist(),truncation=False)['input_ids']])
    show('Panjang token pasangan train',lengths.describe(percentiles=[.5,.9,.95,.99]))
    print('Train melewati PAIR_MAX_LEN (%):',lengths.gt(PAIR_MAX_LEN).mean()*100)
    def encode(frame,labels=True):
        payload={'a':frame.a.tolist(),'b':frame.b.tolist()}
        if labels:payload['labels']=frame.y.astype(float if regression else int).tolist()
        ds=Dataset.from_dict(payload)
        if labels and regression:ds=ds.cast_column('labels',Value('float32'))
        return ds.map(lambda v:tok(v['a'],v['b'],truncation=True,max_length=PAIR_MAX_LEN),batched=True,remove_columns=['a','b'])
    model=AutoModelForSequenceClassification.from_pretrained(path,num_labels=1 if regression else 3,
        problem_type='regression' if regression else 'single_label_classification',ignore_mismatched_sizes=True)
    if not regression:
        model.config.id2label=dict(enumerate(label_names));model.config.label2id={v:i for i,v in enumerate(label_names)}
    def metric(p):
        return {'selection':float(spearmanr(p.label_ids,p.predictions.ravel()).statistic)} if regression else {'selection':f1_score(p.label_ids,p.predictions.argmax(1),average='macro',zero_division=0)}
    args=TrainingArguments(output_dir=str(OUT/'checkpoints'),num_train_epochs=EPOCHS,learning_rate=2e-5,
        per_device_train_batch_size=16,per_device_eval_batch_size=32,fp16=True,eval_strategy='epoch',save_strategy='epoch',
        save_only_model=True,save_total_limit=1,load_best_model_at_end=True,metric_for_best_model='selection',greater_is_better=True,
        report_to='none',logging_strategy='epoch',disable_tqdm=True,seed=SEED)
    trainer=Trainer(model=model,args=args,train_dataset=encode(train),eval_dataset=encode(val),data_collator=DataCollatorWithPadding(tok),compute_metrics=metric)
    trainer.train();a=trainer.predict(encode(val,False)).predictions;b=trainer.predict(encode(test,False)).predictions
    trainer.save_model(str(OUT/'model_indobert'));tok.save_pretrained(str(OUT/'model_indobert'))
    del trainer,model;gc.collect();torch.cuda.empty_cache()
    return (a.ravel(),b.ravel()) if regression else (softmax(a,axis=1),softmax(b,axis=1))

In [8]:
@torch.inference_mode()
def pair_logits(repo,frame,folder=None,max_len=256):
    path=locate(folder,repo) if folder else repo
    tok=AutoTokenizer.from_pretrained(path);model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval()
    mapping={str(k).lower():int(v) for k,v in model.config.label2id.items()};output=[]
    for start in range(0,len(frame),32):
        batch=frame.iloc[start:start+32]
        enc=tok(batch.a.tolist(),batch.b.tolist(),padding=True,truncation=True,max_length=max_len,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):logits=model(**enc).logits
        output.append(logits.float().cpu().numpy())
    del model;gc.collect();torch.cuda.empty_cache()
    return np.concatenate(output),mapping

## 02 · IndoNLI & label mapping eksplisit

Baca JSONL langsung dari repository resmi, tidak menjalankan dataset loading script. Test expert/lay dipertahankan terpisah dalam audit. Endpoint overlap dipurge dari train lalu validation. Pair order tidak ditukar.

In [9]:
CLASSES=['entailment','neutral','contradiction'];mapping={'e':0,'n':1,'c':2}
def load_nli(name,n):
    raw=pd.read_json('https://raw.githubusercontent.com/ir-nlp-csui/indonli/main/data/indonli/'+name+'.jsonl',lines=True)
    assert raw.label.isin(mapping).all(),'Label tidak dikenali; cek sumber.'
    raw['target']=raw.label.map(mapping)
    return prepare_pairs(cap(raw,n),'premise','hypothesis','target')
test=pd.concat([load_nli('test_lay',N_TEST_EACH).assign(source='lay'),load_nli('test_expert',N_TEST_EACH).assign(source='expert')],ignore_index=True)
val=purge(load_nli('val',N_VALID),endpoints(test))
train=purge(load_nli('train',N_TRAIN),endpoints(test)|endpoints(val))
for frame in [train,val,test]:frame['y']=frame.y.astype(int);assert set(frame.y)=={0,1,2}
assert endpoints(train).isdisjoint(endpoints(val)|endpoints(test)) and endpoints(val).isdisjoint(endpoints(test))
print('Train/validation/test:',len(train),len(val),len(test))


Missing sebelum drop


,n,%
pair_id,0,0.0
premise_id,0,0.0
premise,0,0.0
hypothesis,0,0.0
label,0,0.0
data_split,0,0.0
annotator_type,0,0.0
sentence_size,0,0.0
target,0,0.0


Dibuang invalid: 0



Missing sebelum drop


,n,%
pair_id,0,0.0
premise_id,0,0.0
author_label,0,0.0
annotation_round,0,0.0
label,0,0.0
premise,0,0.0
hypothesis,0,0.0
data_split,0,0.0
annotator_type,0,0.0
sentence_size,0,0.0


Dibuang invalid: 0


... (5 outputs left out)

,n,%
pair_id,0,0.0
premise_id,0,0.0
premise,0,0.0
hypothesis,0,0.0
label,0,0.0
data_split,0,0.0
annotator_type,0,0.0
sentence_size,0,0.0
target,0,0.0


Dibuang invalid: 0


Pasangan dikeluarkan karena teks lintas split: 2522


Train/validation/test: 3478 1500 2000


## 03 · EDA wajib NLI

Negasi, angka, arah pasangan, hypothesis-only artifacts, label konflik dan truncation perlu diperhatikan. Tidak melakukan swap augmentation: entailment tidak simetris.

In [10]:
text_eda(train.a,'Premise');text_eda(train.b,'Hypothesis')
show('Kelas train',distribution(train.y.map(dict(enumerate(CLASSES)))))
features=pair_features(train)
show('Fitur per kelas',pd.DataFrame(features,columns=['overlap','premise_only','hyp_only','length_ratio','negation_diff','number_diff']).groupby(train.y).mean())
print('Konflik pair-label:',train.groupby(['a','b']).y.nunique().gt(1).sum())
show('Contoh tiap kelas',train.groupby('y')[['a','b']].head(3))


Premise kualitas


,0
n,3478
null,0
kosong/spasi,0
unik_terisi,1079
duplikat_ekstra_terisi,2399



Premise length


,count,mean,std,min,1%,50%,90%,95%,99%,max
karakter,3478.0,153.8137,115.9696,50.0,62.77,122.0,251.3,328.0,575.0,2179.0
kata,3478.0,21.5745,16.6575,7.0,9.00,17.0,35.0,44.0,79.0,321.0



Premise vocabulary sampel


,0
n_sample,3000.0000
token,63329.0000
unik,6415.0000
hapax,565.0000
hapax_%_tipe,8.8075
hapax_%_token,0.8922


... (32 outputs left out)


Fitur per kelas


,overlap,premise_only,hyp_only,length_ratio,negation_diff,number_diff
y,,,,,,
0,0.2552,0.7121,0.3454,3.0940,0.1138,0.5440
1,0.1677,0.8005,0.5290,3.3492,0.1652,0.6870
2,0.2245,0.7379,0.4267,3.0984,0.5497,0.7166


Konflik pair-label: 0

Contoh tiap kelas


,a,b
0,"Seperti diketahui, saat ini Venna tengah menyandang status janda setelah bercerai dari Ivan Fadilla tahun 2014 silam.",Ivan belum pernah bercerai.
1,Untuk meningkatkan stamina tubuh Anda juga bisa mengonsumsi Fatigon Spirit sebelum memulai aktivitas dan Fatigon Mul...,"Fatigon mengandung vitamin B1, B6 dan B12."
2,Untuk menjelajahi tempat wisata di Bukittinggi ada baiknya untuk menggunakan jasa tour guide agar Anda tak tersesat ...,Tidak ada tour guide yang tersesat.
3,"Fatih Terim digantikan oleh Carlo Ancelotti, meskipun rumor bahwa Franco Baresi akan menjadi manajer baru.",Carlo Ancelotti digantikan Franco baresi.
4,"Steller bahkan menyimpan sebagian dari kulit binatang ini, sehingga ilmuwan bisa mempelajari struktur kulitnya.",Steller ingin ilmuwan mempelajari struktur kulit binatang.
5,"Saat meeting, peserta bisa memanfaatkan filter augmented reality Facebook dan mengubah background menjadi virtual.",Filter augmented reality facebook dapat dimanfaatkan saat meeting.
6,"Tapi kini, justru musuh sekaligus sahabat Batman, Superman tak terlihat batang hidungnya.",Superman tidak kenal batman.
7,"Salad ini terdiri dari selada, apokat, tomat, daging ayam, keju, telur dan bacon.",Tidak ada apokat dalam salad ini.
14,Dari lembaga legislatif dapat digambarkan kegiatan sidang yang dilaksanakan selama tahun 2005 oleh Komisi A sebanyak...,Banyaknya kegiatan sidang yang dilaksanakan selama tahun 2005 meningkat dibanding tahun sebelumnya.


## 04 · Lima model

Mayoritas, pair TF-IDF + LR, E5 pair features + LR, mDeBERTa NLI pretrained, IndoBERT fine-tune tiga kelas. Label model NLI pretrained harus dipetakan lewat config, tidak ditebak dari urutan logits.

In [11]:
from scipy.sparse import hstack
from sklearn.dummy import DummyClassifier
vp={};hp={};y=train.y.to_numpy()
dummy=DummyClassifier(strategy='most_frequent').fit(np.zeros((len(train),1)),y)
vp['01_majority']=dummy.predict_proba(np.zeros((len(val),1)));hp['01_majority']=dummy.predict_proba(np.zeros((len(test),1)))
tf=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=40000,sublinear_tf=True).fit(pd.concat([train.a,train.b]))
def lexical(frame):
    a,b=tf.transform(frame.a),tf.transform(frame.b)
    return hstack([a,b,abs(a-b),a.multiply(b)],format='csr')
lr=LogisticRegression(max_iter=1200,class_weight='balanced',random_state=SEED).fit(lexical(train),y)
vp['02_pair_tfidf_lr']=lr.predict_proba(lexical(val));hp['02_pair_tfidf_lr']=lr.predict_proba(lexical(test))
hyp=LogisticRegression(max_iter=1200,class_weight='balanced',random_state=SEED).fit(tf.transform(train.b),y)
print('Hypothesis-only validation macro-F1:',f1_score(val.y,hyp.predict(tf.transform(val.b)),average='macro'))

Hypothesis-only validation macro-F1: 0.5430502230187528


In [12]:
def dense_features(frame):
    a,b=embed(frame.a),embed(frame.b)
    return np.hstack([a,b,np.abs(a-b),a*b,pair_features(frame)])
model=LogisticRegression(max_iter=1200,class_weight='balanced',random_state=SEED).fit(dense_features(train),y)
vp['03_e5_lr']=model.predict_proba(dense_features(val));hp['03_e5_lr']=model.predict_proba(dense_features(test))
release_encoder()
for frame,target in [(val,vp),(test,hp)]:
    logits,label_map=pair_logits('MoritzLaurer/mDeBERTa-v3-base-mnli-xnli',frame,'mdeberta_nli')
    assert set(CLASSES).issubset(label_map),'Model label mapping tidak eksplisit.'
    target['04_nli_pretrained']=softmax(logits[:,[label_map[k] for k in CLASSES]],axis=1)
vp['05_indobert'],hp['05_indobert']=fine_tune_pairs(train,val,test)


Panjang token pasangan train


,0
count,3478.0000
mean,41.5702
std,24.6070
min,19.0000
50%,36.0000
90%,60.0000
95%,77.0000
99%,117.0000
max,509.0000


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


[transformers] BertForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline/models/indobert
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train melewati PAIR_MAX_LEN (%): 0.14376078205865442


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (5 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': '1.075', 'grad_norm': '24', 'learning_rate': '6.116e-08', 'epoch': '3'}


{'eval_loss': '1.483', 'eval_selection': '0.7062', 'eval_runtime': '5.678', 'eval_samples_per_second': '264.2', 'eval_steps_per_second': '4.227', 'epoch': '3'}


{'train_runtime': '148.5', 'train_samples_per_second': '70.26', 'train_steps_per_second': '2.202', 'train_loss': '1.436', 'epoch': '3'}


## 05 · Validation & post-processing

Ensemble rata-rata dan bias kelas. Bias diuji cross-fit komponen teks validation. Parameter final hanya di-fit validation, bukan test. Prediction test dihitung tanpa label oleh Trainer.

In [13]:
from sklearn.model_selection import StratifiedGroupKFold
def macro(y,p):return f1_score(y,p,labels=[0,1,2],average='macro',zero_division=0)
vp['ensemble']=sum(vp.values())/len(vp);hp['ensemble']=sum(hp.values())/len(hp)
board=pd.Series({name:macro(val.y,p.argmax(1)) for name,p in vp.items()}).sort_values(ascending=False)
show('Validation macro-F1',board);chosen=board.index[0]
def fit_bias(p,y):
    bias=np.zeros(3)
    for _ in range(2):
        for k in range(3):
            best=macro(y,(np.log(np.clip(p,1e-10,1))+bias).argmax(1))
            for value in [-.6,-.3,0,.3,.6]:
                candidate=bias.copy();candidate[k]=value
                score=macro(y,(np.log(np.clip(p,1e-10,1))+candidate).argmax(1))
                if score>best+1e-10:best,bias=score,candidate
    return bias
cross=np.zeros(len(val),int)
cal_groups=pair_groups(val);assert len(np.unique(cal_groups))>=3,'Validation perlu >=3 grup untuk cross-fit.'
for a,b in StratifiedGroupKFold(3,shuffle=True,random_state=SEED).split(val,val.y,cal_groups):
    bias=fit_bias(vp[chosen][a],val.y.to_numpy()[a]);cross[b]=(np.log(np.clip(vp[chosen][b],1e-10,1))+bias).argmax(1)
use_bias=macro(val.y,cross)>board.iloc[0]+.003
bias=fit_bias(vp[chosen],val.y.to_numpy()) if use_bias else np.zeros(3)
(OUT/'selection.json').write_text(json.dumps({'model':chosen,'bias':bias.tolist(),'classes':CLASSES},indent=2))


Validation macro-F1


,0
04_nli_pretrained,0.7966
ensemble,0.7843
05_indobert,0.7062
03_e5_lr,0.6215
02_pair_tfidf_lr,0.5760
01_majority,0.1694


/tmp/ipykernel_697/3948311101.py:22: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  return pd.factorize([root(x) for x in frame.a.map(key)])[0]


152

## 06 · Test lay/expert & kelemahan

Tidak menyimpulkan kelas neutral sebagai tidak entailment saja. Confusion dan error slices memperlihatkan high-overlap contradiction, perubahan angka/negasi, serta pergeseran lay→expert.

In [14]:
final=(np.log(np.clip(hp[chosen],1e-10,1))+bias).argmax(1)
rows=[{'model':name,'macro_f1':macro(test.y,p.argmax(1))} for name,p in hp.items()]+[{'model':'final_selected','macro_f1':macro(test.y,final)}]
show('Test keseluruhan',pd.DataFrame(rows));pd.DataFrame(rows).to_csv(OUT/'models.csv',index=False)
show('Per kelas',pd.DataFrame(classification_report(test.y,final,labels=[0,1,2],target_names=CLASSES,output_dict=True,zero_division=0)).T)
show('Confusion · aktual baris',pd.DataFrame(confusion_matrix(test.y,final,labels=[0,1,2]),index=CLASSES,columns=CLASSES))
audit=test.assign(pred=final,error=final!=test.y,overlap=pair_features(test)[:,0])
show('Lay vs expert',pd.DataFrame([{'source':name,'n':len(g),'macro_f1':macro(g.y,g.pred)} for name,g in audit.groupby('source')]))
for i,label in enumerate(CLASSES):show(label+' errors',audit[(audit.y==i)&audit.error][['a','b','y','pred','source']].head(5))
show('High overlap contradiction',audit[(audit.y==2)&(audit.overlap>.6)][['a','b','pred']].head(TOP))
audit.to_csv(OUT/'predictions.csv',index=False)


Test keseluruhan


,model,macro_f1
0,01_majority,0.1734
1,02_pair_tfidf_lr,0.4851
2,03_e5_lr,0.5467
3,04_nli_pretrained,0.7605
4,05_indobert,0.6170
5,ensemble,0.7182
6,final_selected,0.7670



Per kelas


,precision,recall,f1-score,support
entailment,0.7031,0.8641,0.7753,677.000
neutral,0.7917,0.7113,0.7494,620.000
contradiction,0.8347,0.7255,0.7763,703.000
accuracy,0.7680,0.7680,0.7680,0.768
macro avg,0.7765,0.7670,0.7670,2000.000
weighted avg,0.7768,0.7680,0.7676,2000.000



Confusion · aktual baris


,entailment,neutral,contradiction
entailment,585,57,35
neutral,113,441,66
contradiction,134,59,510


... (6 outputs left out)


contradiction errors


,a,b,y,pred,source
1,Salah satu alat terakhirnya untuk menempatkan tekanan pada Hongaria adalah ancaman mengadakan pemilihan umum yang be...,Hongaria menempatkan tekanan padanya dengan mengancam mengadakan pemilihan umum yang bebas.,2,0,lay
21,Salah satu alat terakhirnya untuk menempatkan tekanan pada Hongaria adalah ancaman mengadakan pemilihan umum yang be...,Ancaman mengadakan pemilihan umum yang bebas merupakan satu-satunya alat terakhir untuk menempatkan tekanan pada Hon...,2,0,lay
22,"Sejak Pesta Olahraga Asia Tenggara dimulai pada tahun 1959, telah diadakan di 15 kota di seluruh Asia Tenggara negar...",Pesta Olahraga Asia telah diadakan di 15 kota di seluruh Asia Tenggara.,2,0,lay
49,"Tatkala membuat penelitian tentang sejarah tari Jawa dan Bali, Edi menggalinya dari data arkeologi.",Edi membuat data arkeologi.,2,0,lay
62,Ada kekhawatiran bahwa mudik dapat membuka peluang bagi virus untuk menyebar ke daerah yang lebih luas.,Kekhawatiran terhadap virus menyebabkan mudik.,2,0,lay



High overlap contradiction


,a,b,pred
117,Sebagian besar penduduk Desa Boya Baliase berada pada usia produktif.,Sebagian besar penduduk Desa Boya Baliase berada pada usia senja.,2
134,Sanurip kemudian dijatuhi hukuman mati pada 23 April 1997.,Sanurip dibebaskan dari hukuman mati pada 23 April 1997.,2
137,Jakarta terletak di pesisir bagian barat laut Pulau Jawa.,Jakarta terletak di bagian timur laut Pulau Jawa.,2
273,Marsudi mengaku tak pernah ada peringatan sejak banjir besar 1 Januari 2020 lalu.,Marsudi mengaku sudah ada peringatan sejak banjir besar 1 Januari 2020 lalu.,2
405,Tiket konser Jay Z disebut murah.,Tiket konser Jay Z disebut mahal.,2
424,"Dengan menyelesaikan jalur maglev ini, saya rasa pemerintah mengharapkan perkembangan ekonomi yang lebih besar di Je...","Dengan hancurnya jalur maglev ini, pemerintah mengharapkan perkembangan ekonomi yang lebih besar di Jepang.",2
472,Seorang investor dengan keuntungan rata-rata dan £10.000 surat obligasi bisa mengharapkan mendapatkan £125 per tahun.,Keuntungan rata-rata dan £10.000 surat obligasi tidak bisa mendapatkan £125 per tahun.,2
482,Salah satunya seorang lelaki yang sedang memakan permen karet yang dengan paksa dikeluarkan dari mulutnya.,Salah satunya seorang perempuan yang sedang memakan permen karet yang dengan paksa dikeluarkan dari mulutnya.,2
507,"Selama tahap-tahap awal Romanisasi, semenanjung ini dibagi dua oleh bangsa Roma untuk tujuan administratif.","Selama tahap awal-awal Romanisasi, semenanjung ini dibagi dua oleh bangsa Roma bukan untuk tujuan administratif.",2
572,"Ketika orang meninggal karena usia tua di India, seharusnya itu adalah sebuah perayaan.","Ketika orang meninggal karena usia tua di India, seharusnya itu bukanlah sebuah perayaan.",2


## Interpretasi & sumber
Semua angka dihitung saat run; pemilihan validation tidak diubah setelah holdout. E5 memakai query: di kedua sisi STS, query:/passage: pada retrieval. [E5](https://huggingface.co/intfloat/multilingual-e5-base), [IndoBERT](https://huggingface.co/indobenchmark/indobert-base-p1), [IndoNLI](https://github.com/ir-nlp-csui/indonli).

In [15]:
import importlib.metadata as metadata
manifest={'task':RESULT_NAME,'seconds':time.perf_counter()-START,'gpu':torch.cuda.get_device_name(0),'seed':SEED,'versions':{p:metadata.version(p) for p in ['torch','transformers','scikit-learn','pandas']}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print(manifest)

{'task': 'hasil_nli3_review', 'seconds': 267.6172478409999, 'gpu': 'Tesla T4', 'seed': 42, 'versions': {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'scikit-learn': '1.6.1', 'pandas': '2.3.3'}}
